# Baseline Error Analysis ? 15/08/2026

This notebook analyzes failure patterns of the locked Historical Simulation and EWMA VaR baselines.

The purpose is diagnostic analysis, not parameter tuning.

## Locked baseline contract

- one-day-ahead VaR
- alpha = 0.05
- Historical Simulation window = 250
- EWMA decay = 0.94
- EWMA evaluation mode = expanding
- strict violation rule: `target_return < quantile_return`
- identical canonical target dates and realized returns
- no use of future observations in forecast interpretation

Canonical input:

`results/baseline_error_analysis.csv`


In [ ]:
import numpy as np
import pandas as pd

DATA_PATH = "results/baseline_error_analysis.csv"

df = pd.read_csv(
    DATA_PATH,
    parse_dates=[
        "forecast_date",
        "target_date",
    ],
)

assert len(df) == 1387
assert df["target_date"].is_unique
assert not df.isna().any().any()

assert int(df["historical_violation"].sum()) == 75
assert int(df["ewma_violation"].sum()) == 73

assert int(df["both_violation"].sum()) == 56
assert int(df["historical_only"].sum()) == 19
assert int(df["ewma_only"].sum()) == 17

print(f"Rows: {len(df)}")
print(
    f"Target range: "
    f"{df['target_date'].iloc[0].date()} "
    f"-> {df['target_date'].iloc[-1].date()}"
)
print(
    f"Historical violations: "
    f"{int(df['historical_violation'].sum())}"
)
print(
    f"EWMA violations: "
    f"{int(df['ewma_violation'].sum())}"
)

## 1. Exception overlap

The two models are evaluated on the same 1,387 target observations.

Exception categories separate days where both models fail from model-specific failures. This avoids comparing violation counts without checking whether the failures occur on the same dates.


In [ ]:
exception_counts = (
    df["exception_type"]
    .value_counts()
    .reindex(
        [
            "both",
            "historical_only",
            "ewma_only",
            "none",
        ]
    )
)

assert exception_counts["both"] == 56
assert exception_counts["historical_only"] == 19
assert exception_counts["ewma_only"] == 17
assert exception_counts["none"] == 1295

print(exception_counts.to_string())

union_exceptions = 56 + 19 + 17
jaccard = 56 / union_exceptions

print(f"Exception union: {union_exceptions}")
print(f"Exception Jaccard overlap: {jaccard:.6%}")

## 2. Exception severity

For each model, exceedance is defined only on a strict violation:

`exceedance = max(0, quantile_return - target_return)`

A larger positive exceedance means the realized loss moved farther below the forecast lower-tail quantile.

This is a diagnostic magnitude, not a new model-selection metric.


In [ ]:
severity = pd.DataFrame(
    {
        "method": [
            "historical",
            "ewma",
        ],
        "exceptions": [
            int(df["historical_violation"].sum()),
            int(df["ewma_violation"].sum()),
        ],
        "mean_exceedance": [
            df.loc[
                df["historical_violation"],
                "historical_exceedance",
            ].mean(),
            df.loc[
                df["ewma_violation"],
                "ewma_exceedance",
            ].mean(),
        ],
        "median_exceedance": [
            df.loc[
                df["historical_violation"],
                "historical_exceedance",
            ].median(),
            df.loc[
                df["ewma_violation"],
                "ewma_exceedance",
            ].median(),
        ],
        "maximum_exceedance": [
            df["historical_exceedance"].max(),
            df["ewma_exceedance"].max(),
        ],
    }
)

assert abs(
    severity.loc[
        severity["method"] == "historical",
        "maximum_exceedance",
    ].iloc[0]
    - 0.052046708010
) <= 1e-12

assert abs(
    severity.loc[
        severity["method"] == "ewma",
        "maximum_exceedance",
    ].iloc[0]
    - 0.052170764728
) <= 1e-12

print(severity.to_string(index=False))

In [ ]:
historical_severe = (
    df.loc[df["historical_violation"]]
    .nlargest(
        10,
        "historical_exceedance",
    )
    [
        [
            "target_date",
            "target_return",
            "historical_quantile",
            "historical_var",
            "historical_exceedance",
            "exception_type",
        ]
    ]
)

ewma_severe = (
    df.loc[df["ewma_violation"]]
    .nlargest(
        10,
        "ewma_exceedance",
    )
    [
        [
            "target_date",
            "target_return",
            "ewma_quantile",
            "ewma_var",
            "ewma_exceedance",
            "exception_type",
        ]
    ]
)

print("Historical top severity days")
print(historical_severe.to_string(index=False))

print()
print("EWMA top severity days")
print(ewma_severe.to_string(index=False))

## 3. Deterministic exception clusters

Cluster adjacency is defined using consecutive observations in the canonical trading-date evaluation sequence, not calendar-day distance.

An isolated exception is therefore a cluster of length 1. A multi-observation cluster has length at least 2.

This definition is fixed before interpreting particular episodes.


In [ ]:
def build_clusters(
    frame,
    violation_col,
    exceedance_col,
    method,
):
    mask = frame[
        violation_col
    ].astype(bool).to_numpy()

    positions = np.flatnonzero(mask)

    records = []
    cluster_id = 0
    start = 0

    for i in range(
        1,
        len(positions) + 1,
    ):
        end_cluster = (
            i == len(positions)
            or positions[i]
            != positions[i - 1] + 1
        )

        if not end_cluster:
            continue

        group = positions[start:i]
        block = frame.iloc[group]

        cluster_id += 1

        records.append(
            {
                "method": method,
                "cluster_id": cluster_id,
                "start_position":
                    int(group[0]),
                "end_position":
                    int(group[-1]),
                "start_date":
                    block[
                        "target_date"
                    ].iloc[0],
                "end_date":
                    block[
                        "target_date"
                    ].iloc[-1],
                "length":
                    len(group),
                "max_exceedance":
                    float(
                        block[
                            exceedance_col
                        ].max()
                    ),
            }
        )

        start = i

    return pd.DataFrame(records)


historical_clusters = build_clusters(
    df,
    "historical_violation",
    "historical_exceedance",
    "historical",
)

ewma_clusters = build_clusters(
    df,
    "ewma_violation",
    "ewma_exceedance",
    "ewma",
)

assert len(historical_clusters) == 63
assert (
    historical_clusters["length"] == 1
).sum() == 53
assert (
    historical_clusters["length"] >= 2
).sum() == 10
assert historical_clusters["length"].max() == 4
assert historical_clusters["length"].sum() == 75

assert len(ewma_clusters) == 65
assert (
    ewma_clusters["length"] == 1
).sum() == 57
assert (
    ewma_clusters["length"] >= 2
).sum() == 8
assert ewma_clusters["length"].max() == 2
assert ewma_clusters["length"].sum() == 73

cluster_summary = pd.DataFrame(
    {
        "method": [
            "historical",
            "ewma",
        ],
        "clusters": [
            63,
            65,
        ],
        "isolated": [
            53,
            57,
        ],
        "multi_observation": [
            10,
            8,
        ],
        "max_length": [
            4,
            2,
        ],
    }
)

print(cluster_summary.to_string(index=False))

In [ ]:
print("Historical largest clusters")

print(
    historical_clusters
    .sort_values(
        [
            "length",
            "max_exceedance",
        ],
        ascending=[
            False,
            False,
        ],
    )
    .head(10)
    .to_string(index=False)
)

print()
print("EWMA largest clusters")

print(
    ewma_clusters
    .sort_values(
        [
            "length",
            "max_exceedance",
        ],
        ascending=[
            False,
            False,
        ],
    )
    .head(10)
    .to_string(index=False)
)

## 4. Temporally aligned risk response

The realized return at target observation `t` is not known when the VaR for that target is produced.

Therefore the correct immediate-response quantity is:

`next_forecast_VaR - current_forecast_VaR`

The notebook first verifies that every target date at row `t` becomes the forecast date at row `t+1`. Only then is the response to the newly observed return analyzed.


In [ ]:
assert np.array_equal(
    df["target_date"]
    .iloc[:-1]
    .to_numpy(),

    df["forecast_date"]
    .iloc[1:]
    .to_numpy(),
)

response = df.iloc[:-1].copy()

response["abs_target_return"] = (
    response["target_return"].abs()
)

for method in [
    "historical",
    "ewma",
]:
    var_col = f"{method}_var"

    response[
        f"{method}_next_var"
    ] = (
        df[var_col]
        .shift(-1)
        .iloc[:-1]
        .to_numpy()
    )

    response[
        f"{method}_next_var_change"
    ] = (
        response[
            f"{method}_next_var"
        ]
        - response[var_col]
    )

print(
    f"Rows with next forecast: "
    f"{len(response)}"
)
print(
    "Every target becomes next forecast date:",
    np.array_equal(
        df["target_date"]
        .iloc[:-1]
        .to_numpy(),

        df["forecast_date"]
        .iloc[1:]
        .to_numpy(),
    ),
)

In [ ]:
top20 = (
    response
    .nlargest(
        20,
        "abs_target_return",
    )
)

response_rows = []

for method in [
    "historical",
    "ewma",
]:
    change_col = (
        f"{method}_next_var_change"
    )

    changes = top20[
        change_col
    ]

    correlation = float(
        response[
            [
                "abs_target_return",
                change_col,
            ]
        ]
        .corr()
        .iloc[0, 1]
    )

    response_rows.append(
        {
            "method": method,
            "top20_increases":
                int(
                    (
                        changes > 0
                    ).sum()
                ),
            "top20_unchanged":
                int(
                    np.isclose(
                        changes,
                        0.0,
                        atol=1e-15,
                    ).sum()
                ),
            "top20_decreases":
                int(
                    (
                        changes < 0
                    ).sum()
                ),
            "mean_next_var_change":
                changes.mean(),
            "median_next_var_change":
                changes.median(),
            "maximum_next_var_increase":
                changes.max(),
            "correlation_abs_return":
                correlation,
        }
    )

response_summary = pd.DataFrame(
    response_rows
)

historical_row = response_summary.loc[
    response_summary["method"]
    == "historical"
].iloc[0]

ewma_row = response_summary.loc[
    response_summary["method"]
    == "ewma"
].iloc[0]

assert historical_row[
    "top20_increases"
] == 13
assert historical_row[
    "top20_unchanged"
] == 7
assert historical_row[
    "top20_decreases"
] == 0

assert ewma_row[
    "top20_increases"
] == 20
assert ewma_row[
    "top20_unchanged"
] == 0
assert ewma_row[
    "top20_decreases"
] == 0

assert abs(
    historical_row[
        "correlation_abs_return"
    ]
    - 0.319946
) <= 1e-6

assert abs(
    ewma_row[
        "correlation_abs_return"
    ]
    - 0.864772
) <= 1e-6

print(response_summary.to_string(index=False))

In [ ]:
top_shocks = (
    response
    .nlargest(
        10,
        "abs_target_return",
    )
    [
        [
            "target_date",
            "target_return",
            "exception_type",
            "historical_var",
            "historical_next_var",
            "historical_next_var_change",
            "ewma_var",
            "ewma_next_var",
            "ewma_next_var_change",
        ]
    ]
)

print(top_shocks.to_string(index=False))

## 5. April 2025 diagnostic episode

April 2025 is inspected because the deterministic cluster analysis identifies the longest Historical exception cluster in the canonical sample.

It is not selected as evidence of a specific market event and no external causal narrative is assigned.


In [ ]:
episode = response.loc[
    (
        response["target_date"]
        >= pd.Timestamp(
            "2025-04-01"
        )
    )
    &
    (
        response["target_date"]
        <= pd.Timestamp(
            "2025-04-10"
        )
    ),
    [
        "target_date",
        "target_return",
        "historical_violation",
        "historical_var",
        "historical_next_var_change",
        "ewma_violation",
        "ewma_var",
        "ewma_next_var_change",
    ],
]

print(episode.to_string(index=False))

## 6. Interpretation

On this canonical sample, the two baseline models exhibit different error dynamics.

Historical Simulation records 75 violations across 63 clusters. Ten clusters contain multiple consecutive evaluation observations, and its longest cluster contains four exceptions.

EWMA records 73 violations across 65 clusters. Eight clusters contain multiple observations and none is longer than two observations.

For the 20 largest absolute realized-return shocks with a following forecast, Historical VaR rises after 13 shocks and remains unchanged after 7. EWMA VaR rises after all 20. The observed association between absolute realized return and next-forecast VaR change is also stronger for EWMA in this sample.

This pattern is consistent with the model mechanics: EWMA directly updates conditional variance after each squared return, while Historical Simulation changes its lower-tail empirical quantile only when the composition and order statistics of the rolling window alter the 5% tail estimate.

The April 2025 episode illustrates this distinction. After the large negative return on 03/04/2025, EWMA VaR rises substantially for the next forecast, while Historical VaR increases much less. Historical subsequently records a four-observation violation cluster.

These are descriptive findings for the audited sample. They do not establish causal market explanations, universal model superiority, or statistical significance.


## 7. Limitations

- The analysis is conditional on the locked canonical baselines.
- No additional window or decay tuning is performed.
- Exception exceedance is a diagnostic quantity rather than an official model-selection metric.
- Cluster adjacency is defined by consecutive evaluation observations rather than calendar days.
- Correlation is descriptive association and is not interpreted causally.
- The April 2025 window is a diagnostic episode identified from the deterministic cluster output, not an external event study.
- Standardized report figures are intentionally outside this notebook and belong to the separate figure-production workflow.


## Prompt provenance

Create Person A's canonical baseline error-analysis notebook from the locked 1,387-row dataset. Cover exception overlap, exceedance severity, deterministic trading-sequence clusters, temporally aligned next-forecast VaR response, and the April-2025 episode. Lock all known canonical counts and outputs with assertions. Do not tune models, alter production code, create standardized figures, stage, or commit. Avoid causal or universal-superiority claims.
